# Geo-Memo Follow-Ups: Which Covariates Predict Regular Transit?

### Head-to-head Random Forests for car access, employment, and ride-share

**Course:** PSYCH 755 · University of Wisconsin–Madison  
**Project:** CA persona / PRCA research framework  

This notebook tests the candidate predictors named in [`memos/geo_predicts_transit.md`](../memos/geo_predicts_transit.md):

1. Car license / access (`Q20` / `Q21`)
2. Employment status
3. Ride-share frequency (`Q28` / `Q29`)
4. Joint mobility bundle (all of the above)

Benchmarks from companion memos: **geo AUC ≈ 0.551**, **CA AUC ≈ 0.590**, chance = 0.500.

Companion memo: [`memos/transit_covariate_followups.md`](../memos/transit_covariate_followups.md)  
CLI: `ca-personas covariate-transit-rf --join inner --seed 42`


## 1. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from ca_personas.load import load_full_cohort
from ca_personas.paths import default_prolific_paths, default_qualtrics_path, sibling_data_available
from ca_personas.transit_covariate_rf import (
    FEATURE_SPECS,
    plot_comparison_memo_figure,
    plot_family_memo_figure,
    run_all_followup_analyses,
    run_feature_family_analysis,
    save_feature_family_artifacts,
    save_followup_bundle,
)

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25})
plt.rcParams["figure.dpi"] = 120

if sibling_data_available():
    PROLIFIC = default_prolific_paths()
    QUALTRICS = default_qualtrics_path()
    source = "../sibling_data File A/B/C"
else:
    staged = Path("/tmp/sibling_data")
    PROLIFIC = [
        staged / "PRCAProlificExport_FileA.csv",
        staged / "PRCAProlificExport_FileB.csv",
    ]
    QUALTRICS = staged / "PRCAQualtricsExport_FileC.csv"
    source = "/tmp/sibling_data File A/B/C"

SEED = 42
N_SPLITS = 5
N_PERM = 30
print("Data source:", source)
print("Prolific:", [str(p) for p in PROLIFIC])
print("Qualtrics:", QUALTRICS)

import json



Data source: /tmp/sibling_data File A/B/C
Prolific: ['/tmp/sibling_data/PRCAProlificExport_FileA.csv', '/tmp/sibling_data/PRCAProlificExport_FileB.csv']
Qualtrics: /tmp/sibling_data/PRCAQualtricsExport_FileC.csv


## 2. Load cohort & run all families


In [2]:
participants, report = load_full_cohort(
    prolific_paths=PROLIFIC,
    qualtrics_path=QUALTRICS,
    join_how="inner",
)
print("Matched analytic rows:", len(participants))
print(report)

bundle = run_all_followup_analyses(
    participants,
    n_splits=N_SPLITS,
    n_perm_repeats=N_PERM,
    random_state=SEED,
)
comparison = bundle["comparison"].copy()
display(comparison)



Matched analytic rows: 241
{'n_prolific_raw': 262, 'n_prolific_unique': 262, 'n_qualtrics_raw': 273, 'n_qualtrics_with_pid': 255, 'n_qualtrics_complete_ca': 260, 'n_joined': 252, 'n_matched_both': 252, 'n_analytic': 241, 'n_dropped_missing_pid': 0, 'n_dropped_incomplete_ca': 11, 'n_dropped_unscorable_ca': 0, 'n_dropped_unjoined': 31, 'n_prolific_only': 10, 'n_qualtrics_only': 21, 'n_qualtrics_missing_pid': 18, 'waves': {'B': 153, 'A': 99}, 'notes': ['Normalized 19 DATA_EXPIRED Student status values to missing.', 'Full Prolific waves (File A/B) omit Ethnicity / Nationality / Language; demos tier uses Age, Sex, Country of residence, and Student status.', 'Analytic sample = Prolific∩Qualtrics with complete scorable PRCA group + interpersonal items.', 'Merge coverage (pre-CA filter): 252 matched Prolific∩Qualtrics; 21 Qualtrics-only (incl. 18 blank Q0 test rows; disregard); 10 Prolific-only (disregard).']}


,spec_key,label,n,n_regular,prevalence,roc_auc,average_precision,balanced_accuracy,f1,brier
3,mobility_bundle,Car + ride-share + employment bundle,143.0,56.0,0.391608,0.746716,0.605258,0.716646,0.666667,0.197594
2,rideshare,Ride-share frequency (Q28/Q29),233.0,99.0,0.424893,0.745138,0.654822,0.731079,0.708520,0.197184
0,car_access,Car license & access (Q20/Q21),149.0,58.0,0.389262,0.607332,0.515021,0.639447,0.476190,0.226675
5,ca_benchmark,Group + interpersonal CA (CA memo benchmark),241.0,101.0,0.419087,0.590000,NaN,NaN,NaN,NaN
4,geo_benchmark,Lat/long (geo memo benchmark),241.0,101.0,0.419087,0.551000,NaN,NaN,NaN,NaN
1,employment,Employment status,241.0,101.0,0.419087,0.528076,0.424449,0.559512,0.585714,0.247240
6,chance,Chance / prevalence,NaN,NaN,NaN,0.500000,NaN,0.500000,NaN,NaN


## 3. Comparison figure


In [3]:
fig_path = ROOT / "memos" / "figures" / "transit_covariate_followups_memo.png"
plot_comparison_memo_figure(comparison, output_path=fig_path)
print("Wrote", fig_path)

for key, analysis in bundle["analyses"].items():
    p = ROOT / "memos" / "figures" / f"{key}_predicts_transit_memo.png"
    plot_family_memo_figure(analysis, output_path=p)
    print(key, "AUC=", f"{analysis['metrics']['roc_auc']:.3f}", "n=", analysis["summary"]["sample"]["n"], "→", p)



Wrote /workspace/memos/figures/transit_covariate_followups_memo.png
car_access AUC= 0.607 n= 149 → /workspace/memos/figures/car_access_predicts_transit_memo.png


employment AUC= 0.528 n= 241 → /workspace/memos/figures/employment_predicts_transit_memo.png
rideshare AUC= 0.745 n= 233 → /workspace/memos/figures/rideshare_predicts_transit_memo.png


mobility_bundle AUC= 0.747 n= 143 → /workspace/memos/figures/mobility_bundle_predicts_transit_memo.png


## 4. Per-family association highlights


In [4]:
for key, analysis in bundle["analyses"].items():
    print("\n===", FEATURE_SPECS[key]["label"], "===")
    print(analysis["summary"]["verdict"]["interpretation"])
    display(analysis["associations"])




=== Car license & access (Q20/Q21) ===
Car license & access (Q20/Q21) Random Forest CV ROC-AUC = 0.607. Stronger discrimination than the geo benchmark (≈0.55). Exceeds the CA-score benchmark (≈0.59).


,feature,level,n,n_regular,n_not_regular,pct_regular
1,Q20,Yes,129,47,82,0.364341
0,Q20,No,20,11,9,0.550000
4,Q21,Yes,123,38,85,0.308943
2,Q21,No,25,19,6,0.760000
3,Q21,Not Sure,1,1,0,1.000000



=== Employment status ===
Employment status Random Forest CV ROC-AUC = 0.528. Comparable to or weaker than the geo benchmark (≈0.55). Does not exceed the CA-score benchmark (≈0.59).


,feature,level,n,n_regular,n_not_regular,pct_regular
0,Employment status,Full-Time,148,67,81,0.452703
1,Employment status,Other,62,19,43,0.306452
2,Employment status,Part-Time,31,15,16,0.483871



=== Ride-share frequency (Q28/Q29) ===
Ride-share frequency (Q28/Q29) Random Forest CV ROC-AUC = 0.745. Stronger discrimination than the geo benchmark (≈0.55). Exceeds the CA-score benchmark (≈0.59).


,feature,level,n,n_regular,n_not_regular,pct_regular
1,Q28,2-4 days a month,65,34,31,0.523077
4,Q28,Never,62,9,53,0.145161
0,Q28,0-1 days a month,47,11,36,0.234043
2,Q28,4-8 days a month,42,29,13,0.690476
3,Q28,8 or more days a month,17,16,1,0.941176
5,Q29,1-2 rides in a typical day,206,82,124,0.398058
6,Q29,3-4 rides in a typical day,17,10,7,0.588235
7,Q29,5-6 rides in a typical day,9,6,3,0.666667
8,Q29,7 or more rides in a typical day,1,1,0,1.000000



=== Car + ride-share + employment bundle ===
Car + ride-share + employment bundle Random Forest CV ROC-AUC = 0.747. Stronger discrimination than the geo benchmark (≈0.55). Exceeds the CA-score benchmark (≈0.59).


,feature,level,n,n_regular,n_not_regular,pct_regular
13,Employment status,Full-Time,81,37,44,0.456790
14,Employment status,Other,45,14,31,0.311111
15,Employment status,Part-Time,17,5,12,0.294118
1,Q20,Yes,123,45,78,0.365854
0,Q20,No,20,11,9,0.550000
4,Q21,Yes,118,37,81,0.313559
2,Q21,No,24,18,6,0.750000
3,Q21,Not Sure,1,1,0,1.000000
9,Q28,Never,48,9,39,0.187500
6,Q28,2-4 days a month,42,24,18,0.571429


## 5. Export artifacts


In [5]:
out = ROOT / "outputs" / "transit_covariate_rf"
paths = save_followup_bundle(bundle, out)
print("Results card:", paths["results_card"])
print(json.dumps(bundle["results_card"], indent=2)[:1500])



Results card: /workspace/outputs/transit_covariate_rf/followup_results_card.json
{
  "secondary_rq": "Among the geo-memo follow-up candidates (car access/license, employment, ride-share frequency), which features best predict regular public-transit use?",
  "benchmarks": {
    "geo_auc": 0.551,
    "ca_auc": 0.59,
    "chance_auc": 0.5
  },
  "best_spec": "mobility_bundle",
  "best_auc": 0.7467159277504106,
  "comparison": [
    {
      "spec_key": "mobility_bundle",
      "label": "Car + ride-share + employment bundle",
      "n": 143.0,
      "n_regular": 56.0,
      "prevalence": 0.3916083916083916,
      "roc_auc": 0.7467159277504106,
      "average_precision": 0.6052583001385728,
      "balanced_accuracy": 0.7166461412151067,
      "f1": 0.6666666666666666,
      "brier": 0.19759361412167553
    },
    {
      "spec_key": "rideshare",
      "label": "Ride-share frequency (Q28/Q29)",
      "n": 233.0,
      "n_regular": 99.0,
      "prevalence": 0.4248927038626609,
      "roc_auc":

## 6. Interpretation notes

- Ride-share days (`Q28`) carries the strongest single-family signal among the geo-memo candidates.
- The joint bundle is essentially ride-share-dominated on the complete-case subset that also has car items.
- Car access improves on geography/CA but is limited by missingness on `Q20`/`Q21`.
- Employment status alone is near chance and weaker than the geo RF.
